# Predicting Loan Default: A Machine Learning Approach to Credit Risk Assessment

**Author:** Atilla Ahmed
**Dataset:** Lending Club Loan Data (2007–2018), ~2.26M loans, 145 features

---

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Introduction and Problem Formulation](#2-introduction-and-problem-formulation)
3. [Data Loading and Initial Inspection](#3-data-loading-and-initial-inspection)
4. [Data Cleaning and Preprocessing](#4-data-cleaning-and-preprocessing)
5. [Exploratory Data Analysis](#5-exploratory-data-analysis)
6. [Feature Engineering & Preprocessing Pipeline](#6-feature-engineering)
7. [Mathematical Framework](#7-mathematical-framework)
8. [Baseline Models](#8-baseline-models)
9. [Advanced Models and Hyperparameter Tuning](#9-advanced-models-and-hyperparameter-tuning)
10. [Model Evaluation and Comparison](#10-model-evaluation-and-comparison)
11. [Business Impact Analysis](#11-business-impact-analysis)
12. [Conclusions, Limitations, and Future Work](#12-conclusions-limitations-and-future-work)
13. [References](#13-references)

## 1. Setup and Imports <a id="1-setup-and-imports"></a>

All libraries and global settings are defined in one place so the notebook is fully reproducible. The random seed is fixed at 42 throughout.

In [2]:
# SETUP — libraries, global config, random seed

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix, classification_report,
                              roc_curve, precision_recall_curve)
from scipy.stats import randint, uniform

SEED = 42
np.random.seed(SEED)


## 2. Introduction and Problem Formulation <a id="2-introduction-and-problem-formulation"></a>

### 2.1 Background

Lending Club was the largest peer-to-peer lending platform in the US. Investors lend money directly to borrowers, and the central risk they face is credit default a borrower stops repaying. When that happens the investor loses part or all of their principal. Predicting which borrowers will default is therefore critical for anyone deploying capital on the platform.

This project builds a classification model to predict loan default using only information available at the time of loan issuance. No post-origination data is used.

### 2.2 Problem Statement

**Task:** Binary classification - *Fully Paid* (0) vs *Charged Off / Default* (1).

**Target:** `loan_status`, filtered to loans with a known final outcome only.

**Key constraint:** Only features available at origination. Any variable that reflects what happened during repayment is excluded using those would be data leakage and would produce a model that works perfectly in training but fails entirely on new applications.

**Evaluation metric:** Accuracy is misleading on imbalanced datasets. Missing a default costs real money; rejecting a good borrower costs opportunity. This asymmetry makes Recall and PR-AUC far more relevant.

### 2.3 Expected Loss Framework

$$\text{Expected Loss} = PD \times LGD \times EAD$$

- **PD (Probability of Default)** - the likelihood a borrower fails to repay. This is what the model predicts.
- **LGD (Loss Given Default)** - the fraction of the outstanding amount that is not recovered. Typically 40–60% for unsecured consumer loans.
- **EAD (Exposure at Default)** - the total amount owed at the time of default.

The model targets the PD component. A well-calibrated PD estimate allows a lender to price risk correctly and make informed accept/reject decisions on new applications.

## 3. Data Loading and Initial Inspection <a id="3-data-loading-and-initial-inspection"></a>

The dataset is a single 1.1 GB CSV file with 2.26 million loan records and 145 columns. A data dictionary is included as a separate Excel file.

In [3]:
loans = pd.read_csv('loan.csv', low_memory=False)
print(f'Shape: {loans.shape[0]:,} rows x {loans.shape[1]} columns')

Shape: 2,260,668 rows x 145 columns


In [4]:
loans.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
loans.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 145 columns):
 #    Column                                      Non-Null Count    Dtype  
---   ------                                      --------------    -----  
 0    id                                          0 non-null        float64
 1    member_id                                   0 non-null        float64
 2    loan_amnt                                   2260668 non-null  int64  
 3    funded_amnt                                 2260668 non-null  int64  
 4    funded_amnt_inv                             2260668 non-null  float64
 5    term                                        2260668 non-null  str    
 6    int_rate                                    2260668 non-null  float64
 7    installment                                 2260668 non-null  float64
 8    grade                                       2260668 non-null  str    
 9    sub_grade                                   2260668 non

### 3.1 Target Variable Distribution

Before doing anything else I need to understand what `loan_status` contains and decide which values map to default vs non-default.

In [6]:
(loans['loan_status'].value_counts(normalize=True).round(4) * 100)

loan_status
Fully Paid                                             46.09
Current                                                40.68
Charged Off                                            11.57
Late (31-120 days)                                      0.97
In Grace Period                                         0.40
Late (16-30 days)                                       0.17
Does not meet the credit policy. Status:Fully Paid      0.09
Does not meet the credit policy. Status:Charged Off     0.03
Default                                                 0.00
Name: proportion, dtype: float64

Looking at the distribution, a few things stand out:

- **Current (40.7%)** - these loans are still active. The outcome is unknown, so they have to be excluded.
- **Fully Paid (46.1%)** - borrower repaid in full → class 0.
- **Charged Off (11.6%)** - lender wrote off the debt → class 1.
- **Late / In Grace Period (1.5%)** - delinquent but not yet charged off. Ambiguous label, so excluded to avoid noise.
- The remaining categories (`Does not meet credit policy`, `Default`) are either negligible or will be mapped to their obvious outcome.

I keep only *Fully Paid* and *Charged Off* as the two definitive outcomes. Everything else is excluded.

### 3.2 Missing Values Overview

With 145 columns there will be significant gaps. This analysis is run on the raw dataset to get a baseline before any filtering. Worth noting: the effective missing rate for some columns will shift after I filter to known outcomes in Section 4, which is why I re-check missing values on the filtered data before deciding what to drop.

In [7]:
missing_raw     = loans.isna().sum()
missing_pct_raw = (missing_raw / len(loans) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing' : missing_raw,
    'Percent' : missing_pct_raw
}).sort_values('Percent', ascending=False)

missing_df[missing_df['Missing'] > 0]

,Missing,Percent
id,2260668,100.00
url,2260668,100.00
member_id,2260668,100.00
orig_projected_additional_accrued_interest,2252242,99.63
hardship_length,2250055,99.53
...,...,...
acc_now_delinq,29,0.00
last_credit_pull_d,73,0.00
tax_liens,105,0.00
total_acc,29,0.00


Quick summary of how severe the missingness is:

In [8]:
total_cols = loans.shape[1]
empty_cols = (missing_pct_raw == 100).sum()
over_90    = (missing_pct_raw > 90).sum()
over_50    = (missing_pct_raw > 50).sum()

print(f'Total columns:         {total_cols}')
print(f'Completely empty:      {empty_cols}')
print(f'Over 90% missing:      {over_90}')
print(f'Over 50% missing:      {over_50}')

Total columns:         145
Completely empty:      3
Over 90% missing:      38
Over 50% missing:      44


Three columns are completely empty - `id`, `url`, `member_id` - stripped for privacy. The 38 columns above 90% missing are mostly hardship and settlement details that only exist for a small fraction of borrowers who entered those programs. The 44 columns above 50% include joint application fields populated only for co-borrowers, which is about 5% of loans.

Columns above 50% missing will be dropped, but the threshold will be re-evaluated after filtering.

## 4. Data Cleaning and Preprocessing <a id="4-data-cleaning-and-preprocessing"></a>

The raw dataset needs several rounds of cleaning before it is usable. I follow a deliberate order: filter first, then recheck missing rates, then drop, then convert types, then impute. Doing it in any other sequence can produce incorrect missing percentages or introduce subtle errors.